In [1]:
# Core data handling and plotting libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import holidays  # for German public holiday flags (Bavaria has some state-specific ones)

# Consistent plot styling with Notebook 1
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 5)

STATE = "BY"  # Bavaria — change this single variable to re-run on another state

In [2]:
# Load the cleaned, all-states demand table produced by Notebook 1
# (much faster than re-parsing the 66MB raw file, and already has the 0 MWh glitch fixed)
demand_all = pd.read_csv("../data/demand_by_state_cleaned.csv", index_col=0, parse_dates=True)
demand_all.index.name = "datetime"

# Pull out just our target state as a single series
demand = demand_all[STATE].copy()
demand.name = "demand_mwh"

print(demand.shape)
print(demand.head())

(35064,)
datetime
2019-01-01 00:00:00    9273.3600
2019-01-01 01:00:00    7028.2110
2019-01-01 02:00:00    6659.5195
2019-01-01 03:00:00    6466.1016
2019-01-01 04:00:00    6418.0590
Name: demand_mwh, dtype: float64


In [3]:
# Build a features DataFrame indexed the same way as our demand series
features = pd.DataFrame(index=demand.index)

# Basic calendar features extracted directly from the datetime index
features["hour"] = features.index.hour              # 0-23, captures the midday plateau we found
features["day_of_week"] = features.index.dayofweek  # 0=Monday ... 6=Sunday
features["month"] = features.index.month             # 1-12, captures winter-peak seasonality
features["year"] = features.index.year                # useful for spotting yearly effects (COVID dip, 2022 dip)

# Weekend flag — Saturday=5, Sunday=6 — since we saw a ~23% demand drop on weekends
features["is_weekend"] = (features["day_of_week"] >= 5).astype(int)

# German public holidays for Bavaria specifically (has a few extra Catholic holidays
# vs. other German states, e.g. Epiphany, Corpus Christi)
de_holidays = holidays.Germany(state="BY", years=range(2019, 2023))
features["is_holiday"] = features.index.normalize().isin(de_holidays).astype(int)

print(features.head())
print("\nHoliday count in dataset:", features["is_holiday"].sum(), "hours")
print("Unique holidays flagged:", features.index[features["is_holiday"] == 1].normalize().nunique())

                     hour  day_of_week  month  year  is_weekend  is_holiday
datetime                                                                   
2019-01-01 00:00:00     0            1      1  2019           0           0
2019-01-01 01:00:00     1            1      1  2019           0           0
2019-01-01 02:00:00     2            1      1  2019           0           0
2019-01-01 03:00:00     3            1      1  2019           0           0
2019-01-01 04:00:00     4            1      1  2019           0           0

Holiday count in dataset: 0 hours
Unique holidays flagged: 0


In [4]:
# Build a features DataFrame indexed the same way as our demand series
features = pd.DataFrame(index=demand.index)

# Basic calendar features extracted directly from the datetime index
features["hour"] = features.index.hour              # 0-23, captures the midday plateau we found
features["day_of_week"] = features.index.dayofweek  # 0=Monday ... 6=Sunday
features["month"] = features.index.month             # 1-12, captures winter-peak seasonality
features["year"] = features.index.year                # useful for spotting yearly effects (COVID dip, 2022 dip)

# Weekend flag — Saturday=5, Sunday=6 — since we saw a ~23% demand drop on weekends
features["is_weekend"] = (features["day_of_week"] >= 5).astype(int)

# German public holidays for Bavaria specifically (has a few extra Catholic holidays
# vs. other German states, e.g. Epiphany, Corpus Christi)
de_holidays = holidays.Germany(state="BY", years=range(2019, 2023))

# Convert the index to plain date objects (not Timestamps) to match how the
# holidays library stores its dates internally — .isin() with Timestamps
# silently fails to match against date objects, which is what caused 0 hits initially
features["is_holiday"] = pd.Series(
    [d in de_holidays for d in features.index.date],
    index=features.index
).astype(int)

print(features.head())
print("\nHoliday count in dataset:", features["is_holiday"].sum(), "hours")
print("Unique holidays flagged:", features.index[features["is_holiday"] == 1].normalize().nunique())

                     hour  day_of_week  month  year  is_weekend  is_holiday
datetime                                                                   
2019-01-01 00:00:00     0            1      1  2019           0           1
2019-01-01 01:00:00     1            1      1  2019           0           1
2019-01-01 02:00:00     2            1      1  2019           0           1
2019-01-01 03:00:00     3            1      1  2019           0           1
2019-01-01 04:00:00     4            1      1  2019           0           1

Holiday count in dataset: 1152 hours
Unique holidays flagged: 48


In [5]:
# Cyclical (sine/cosine) encoding for hour, day-of-week, and month.
# Raw integers make hour=23 and hour=0 look far apart to a model, when they're
# actually adjacent — sin/cos encoding preserves that "wraparound" relationship.

# Hour of day (period = 24)
features["hour_sin"] = np.sin(2 * np.pi * features["hour"] / 24)
features["hour_cos"] = np.cos(2 * np.pi * features["hour"] / 24)

# Day of week (period = 7)
features["dow_sin"] = np.sin(2 * np.pi * features["day_of_week"] / 7)
features["dow_cos"] = np.cos(2 * np.pi * features["day_of_week"] / 7)

# Month of year (period = 12)
features["month_sin"] = np.sin(2 * np.pi * features["month"] / 12)
features["month_cos"] = np.cos(2 * np.pi * features["month"] / 12)

print(features[["hour", "hour_sin", "hour_cos", "day_of_week", "dow_sin", "dow_cos"]].head())

                     hour  hour_sin  hour_cos  day_of_week   dow_sin  dow_cos
datetime                                                                     
2019-01-01 00:00:00     0  0.000000  1.000000            1  0.781831  0.62349
2019-01-01 01:00:00     1  0.258819  0.965926            1  0.781831  0.62349
2019-01-01 02:00:00     2  0.500000  0.866025            1  0.781831  0.62349
2019-01-01 03:00:00     3  0.707107  0.707107            1  0.781831  0.62349
2019-01-01 04:00:00     4  0.866025  0.500000            1  0.781831  0.62349


In [6]:
# Lag features: demand at fixed offsets in the past, aligned to the current timestamp.
# These are usually the strongest predictors in load forecasting since demand
# is highly autocorrelated (today resembles yesterday, and even more so last week).

# Same hour, 24 hours ago (yesterday)
features["lag_24h"] = demand.shift(24)

# Same hour, 168 hours ago (exactly one week ago — same day-of-week too, not just same hour)
features["lag_168h"] = demand.shift(168)

# Rolling statistics: smoothed recent behavior rather than a single past point.
# closed="left" excludes the current hour itself, so we never leak the value
# we're trying to predict into its own feature.
features["rolling_mean_24h"] = demand.shift(1).rolling(window=24).mean()
features["rolling_std_24h"] = demand.shift(1).rolling(window=24).std()
features["rolling_mean_168h"] = demand.shift(1).rolling(window=168).mean()

# These lag/rolling features are NaN for the first 168 hours of the dataset
# (not enough history yet) — check how many rows that affects
print("Rows with any NaN from lag/rolling features:", features.isna().any(axis=1).sum())
print("\nFirst valid row (all lag features populated):")
print(features.dropna().index.min())
print("\nSample of populated rows:")
print(features[["lag_24h", "lag_168h", "rolling_mean_24h", "rolling_std_24h", "rolling_mean_168h"]].dropna().head())

Rows with any NaN from lag/rolling features: 168

First valid row (all lag features populated):
2019-01-08 00:00:00

Sample of populated rows:
                       lag_24h   lag_168h  rolling_mean_24h  rolling_std_24h  \
datetime                                                                       
2019-01-08 00:00:00  7662.0750  9273.3600      10733.824029      1899.096151   
2019-01-08 01:00:00  7646.8813  7028.2110      10800.960904      1812.278878   
2019-01-08 02:00:00  7492.1455  6659.5195      10863.102225      1722.785846   
2019-01-08 03:00:00  7542.2134  6466.1016      10921.068621      1625.065422   
2019-01-08 04:00:00  7827.1970  6418.0590      10978.436021      1521.609174   

                     rolling_mean_168h  
datetime                                
2019-01-08 00:00:00        9376.061568  
2019-01-08 01:00:00        9376.061568  
2019-01-08 02:00:00        9388.621461  
2019-01-08 03:00:00        9401.858482  
2019-01-08 04:00:00        9416.459252  


In [7]:
# Attach the target (what we're actually predicting) to the same DataFrame
features["demand_mwh"] = demand

# Drop the first 168 hours — they can't have a valid lag_168h feature,
# so they're unusable for training regardless of the model
features_clean = features.dropna().copy()

print("Shape before dropna:", features.shape)
print("Shape after dropna:", features_clean.shape)
print("\nDate range:", features_clean.index.min(), "to", features_clean.index.max())
print("\nColumns:", features_clean.columns.tolist())

Shape before dropna: (35064, 18)
Shape after dropna: (34896, 18)

Date range: 2019-01-08 00:00:00 to 2022-12-31 23:00:00

Columns: ['hour', 'day_of_week', 'month', 'year', 'is_weekend', 'is_holiday', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_std_24h', 'rolling_mean_168h', 'demand_mwh']


In [8]:
# Correlation of each feature with the demand target — sanity check that our
# engineered features actually carry the signal we expect from Notebook 1's EDA
correlations = features_clean.corr()["demand_mwh"].drop("demand_mwh").sort_values(key=abs, ascending=False)
print(correlations)

lag_168h             0.926507
lag_24h              0.805658
rolling_mean_24h     0.600728
hour_cos            -0.582224
is_weekend          -0.448758
rolling_mean_168h    0.435357
dow_sin              0.398931
day_of_week         -0.394907
rolling_std_24h      0.350617
hour                 0.348773
month_cos            0.295895
hour_sin            -0.253967
month_sin            0.186154
is_holiday          -0.154696
month               -0.133817
dow_cos             -0.059461
year                -0.025197
Name: demand_mwh, dtype: float64


In [9]:
# Time-based split — NOT a random split, which would leak future information
# into training via adjacent hours. Train on everything through mid-2022,
# hold out the last 6 months (Jul-Dec 2022) as an honest validation set.
split_date = "2022-07-01"

train = features_clean[features_clean.index < split_date]
val = features_clean[features_clean.index >= split_date]

print("Train shape:", train.shape, "|", train.index.min(), "to", train.index.max())
print("Val shape:  ", val.shape, "|", val.index.min(), "to", val.index.max())
print(f"\nTrain: {len(train)/len(features_clean)*100:.1f}% | Val: {len(val)/len(features_clean)*100:.1f}%")

Train shape: (30480, 18) | 2019-01-08 00:00:00 to 2022-06-30 23:00:00
Val shape:   (4416, 18) | 2022-07-01 00:00:00 to 2022-12-31 23:00:00

Train: 87.3% | Val: 12.7%


In [10]:
# Save train/validation feature sets separately so Notebook 3 (modeling) doesn't
# need to repeat any of this feature engineering
train.to_csv("../data/train_features.csv")
val.to_csv("../data/val_features.csv")

print("Saved train_features.csv:", train.shape)
print("Saved val_features.csv:", val.shape)

Saved train_features.csv: (30480, 18)
Saved val_features.csv: (4416, 18)


## Summary

- Loaded the cleaned Bavaria demand series from Notebook 1 and built a full feature set:
  - **Calendar features:** hour, day-of-week, month, year, weekend flag, and a Bavaria-specific
    public holiday flag (48 unique holidays flagged across 2019–2022, ~12/year)
  - **Cyclical encoding:** sin/cos transforms of hour, day-of-week, and month, so the model sees
    hour 23 and hour 0 as adjacent rather than maximally distant
  - **Lag features:** demand 24 hours ago (yesterday, same hour) and 168 hours ago (same hour, one
    week ago)
  - **Rolling statistics:** 24-hour and 168-hour rolling mean, plus 24-hour rolling std — all
    shifted by 1 hour first so the current hour's own value never leaks into its own features
- Fixed a real bug along the way: the `holidays` library stores dates as plain `date` objects, not
  `Timestamp`s — comparing directly against a `Timestamp`-based index silently matched zero holidays
  until fixed
- **Correlation check confirmed the engineered features carry real signal**, matching Notebook 1's
  findings: `lag_168h` is the strongest single predictor (r = 0.93), followed by `lag_24h` (r = 0.81);
  `is_weekend` is meaningfully negative (r = -0.45), consistent with the ~23% weekend demand drop
  found in the EDA
- Dropped the first 168 hours of the dataset (Jan 1–7, 2019) since `lag_168h` has no valid history
  before that point — final feature set spans **2019-01-08 to 2022-12-31**
- **Time-based split** (not random — avoids leaking future information into training):
  - **Train:** 30,480 rows, 2019-01-08 to 2022-06-30 (87.3%)
  - **Validation:** 4,416 rows, 2022-07-01 to 2022-12-31 (12.7%, last 6 months)
- Saved both sets to `../data/train_features.csv` and `../data/val_features.csv` for Notebook 3

**Next:** Notebook 3 — train and compare Linear Regression, Random Forest/XGBoost, and SARIMAX on
the validation set, then retrain the winning model on the full dataset and generate a genuine forward
forecast into 2023 for later real-world validation.